%md
# Error Handling Demo — Databricks Notebook Logger

This notebook demonstrates how the logger behaves when a notebook encounters an error. All executed code, outputs, warnings, and the error message are automatically captured in the log file before logging stops. The log file is exported to the Databricks Workspace and both the log file and HTML snapshot to your designated SFTP location.

### Start Logging
Initalize the logger and proceed with notebook code as normal

In [0]:
# Install and import the logging module
# %pip install notebook-logger
# from notebook_logger import *

import sys
if "/Workspace/Repos/hmcooper@berkeley.edu/Databricks-Notebook-Logger/src" not in sys.path:
    sys.path.insert(0, "/Workspace/Repos/hmcooper@berkeley.edu/Databricks-Notebook-Logger/src")
from notebook_logger import *

In [0]:
start_logging(output_directory="/output/logs")

In [0]:
# Imports
import pyspark
from pyspark import sql, SparkConf
from pyspark.sql import functions as f
from pyspark.sql.types import *
import os, sys
from pyspark.sql.window import Window
import warnings
from datetime import date

In [0]:
# Create sample patient visit data
visit_data = [
    ("P001", date(2023, 8, 21), "99213", "Hypertension"),
    ("P002", date(2023, 8, 22), "99214", "Diabetes"),
    ("P003", date(2023, 8, 23), "99212", "Annual checkup"),
    ("P001", date(2023, 8, 24), "99215", "Hypertension"),
    ("P004", date(2023, 8, 25), "99213", "Asthma"),
    ("P002", date(2023, 8, 26), "99213", "Diabetes"),
]

patient_visits = spark.createDataFrame(visit_data, ["patient_id", "visit_date", "procedure_code", "diagnosis"])
patient_visits.show()

## Error Handling
Upon encountering an error in the notebook, the logger immediately stops execution, appends a full error summary to the log, and uploads both the log file and HTML snapshot to the Workspace and SFTP location - ensuring a complete audit trail is preserved even when a notebook fails

In [0]:
raise ValueError("This is a test error.")

In [0]:
# Create sample patient demographics table
demographics_data = [
    ("P001", "Male", 45, "Commercial"),
    ("P002", "Female", 62, "Medicare"),
    ("P003", "Male", 34, "Commercial"),
    ("P004", "Female", 28, "Medicaid")
]

demographics = spark.createDataFrame(demographics_data, ["patient_id", "gender", "age", "insurance"])
demographics.show()

In [0]:
# Join sample visit with demographic data
patient_analysis = patient_visits.join(demographics, on="patient_id", how="left")
patient_analysis.show()

In [0]:
# Create sample summary statistics by demographics
summary = patient_analysis.groupBy("gender", "insurance") \
    .agg(
        f.count("visit_date").alias("total_visits"),
        f.countDistinct("patient_id").alias("unique_patients"),
        f.avg("age").alias("avg_age")
    ) \
    .orderBy("gender", "insurance")

summary.show(truncate=False)

In [0]:
# Create temporary view to query with SQL
patient_visits.createOrReplaceTempView("patient_visits")

In [0]:
# Query from spark.sql() will show up in the log
spark.sql("""
create or replace temp view diagnosis_summary as
select diagnosis, count(*) as patient_count
from patient_visits
group by diagnosis
""")

In [0]:
# Use .show(truncate=False) to execute query and display output in the log file
# truncate=False ensures no values are truncated/cut off in the log file
spark.sql("""
select *
from diagnosis_summary
where diagnosis = 'Hypertension'
    or diagnosis = 'Diabetes'
""").show(truncate=False)

In [0]:
# Use log_df to display table contents in the log file
log_df("diagnosis_summary", limit=10)

In [0]:
# User warning
warnings.warn("This is a test warning", UserWarning)

In [0]:
stop_logging()